In [2]:
from pathlib import Path

import pandas as pd


PROJECT_ROOT = Path.cwd()

FEATURE_FILE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "candidate_point_features.csv"
)

df = pd.read_csv(FEATURE_FILE, encoding="utf-8-sig")

print("Dataset shape:", df.shape)
print("\nTotal candidate locations:", len(df))

print("\nFeature columns:")
print(df.columns.tolist())

print("\nMissing values:")
display(df.isna().sum().sort_values(ascending=False))

print("\nFirst five records:")
display(df.head())

Dataset shape: (1002, 24)

Total candidate locations: 1002

Feature columns:
['location_id', 'longitude', 'latitude', 'zone_name', 'population_2011', 'dist_road_m', 'dist_traffic_m', 'dist_hospital_m', 'dist_college_m', 'dist_school_m', 'dist_market_m', 'dist_food_hotel_m', 'dist_finance_m', 'dist_vehicle_service_m', 'dist_waterway_m', 'dist_waterbody_m', 'building_count_500m', 'poi_count_500m', 'health_poi_count_500m', 'commercial_poi_count_500m', 'land_use_class', 'elevation_m', 'slope_deg', 'flood_risk_proxy']

Missing values:


location_id                  0
longitude                    0
latitude                     0
zone_name                    0
population_2011              0
dist_road_m                  0
dist_traffic_m               0
dist_hospital_m              0
dist_college_m               0
dist_school_m                0
dist_market_m                0
dist_food_hotel_m            0
dist_finance_m               0
dist_vehicle_service_m       0
dist_waterway_m              0
dist_waterbody_m             0
building_count_500m          0
poi_count_500m               0
health_poi_count_500m        0
commercial_poi_count_500m    0
land_use_class               0
elevation_m                  0
slope_deg                    0
flood_risk_proxy             0
dtype: int64


First five records:


,location_id,longitude,latitude,zone_name,population_2011,dist_road_m,dist_traffic_m,dist_hospital_m,dist_college_m,dist_school_m,...,dist_waterway_m,dist_waterbody_m,building_count_500m,poi_count_500m,health_poi_count_500m,commercial_poi_count_500m,land_use_class,elevation_m,slope_deg,flood_risk_proxy
0,1,73.756295,18.531983,Admin Ward 01 Aundh,180264.0,63.912745,2554.453161,2656.654669,2756.862746,2762.467381,...,2225.427074,2355.857536,25,0,0,0,forest,657.0,13.665863,Low
1,2,73.761031,18.532014,Admin Ward 01 Aundh,180264.0,8.326502,2182.850969,2236.843921,2342.377309,2300.901381,...,1756.318474,1867.756572,92,0,0,0,forest,631.0,5.602115,Low
2,3,73.760999,18.536532,Admin Ward 01 Aundh,180264.0,96.469324,1854.429628,2179.448439,2064.879739,2271.005033,...,1833.781162,2047.747091,61,0,0,0,forest,662.0,10.661363,Low
3,4,73.766059,18.491383,Admin Ward 03 Kothrud Karveroad,209046.0,7.424411,2405.992842,2412.172555,1286.193628,1800.148457,...,1531.198229,3355.323774,7,0,0,0,unknown,742.0,5.913284,Low
4,5,73.765995,18.500419,Admin Ward 03 Kothrud Karveroad,209046.0,47.504002,1614.152389,1549.490810,476.130858,859.729440,...,630.547047,2507.593358,57,1,0,0,unknown,702.0,7.080391,Medium


In [3]:

from pathlib import Path

print("Current working directory:", Path.cwd())
print("Parent directory:", Path.cwd().parent)

Current working directory: C:\Users\jamad\Documents\geosense
Parent directory: C:\Users\jamad\Documents


In [4]:
# Step 1: Create population_level from ward population

population = df["population_2011"].copy()

low_threshold = population.quantile(0.33)
high_threshold = population.quantile(0.67)

def get_population_level(pop):
    if pop <= low_threshold:
        return "Low"
    elif pop >= high_threshold:
        return "High"
    else:
        return "Medium"

df["population_level"] = df["population_2011"].apply(get_population_level)

print("Population level distribution:")
display(df["population_level"].value_counts())

print("\nLow threshold:", low_threshold)
print("High threshold:", high_threshold)

Population level distribution:


population_level
Low       387
High      387
Medium    228
Name: count, dtype: int64


Low threshold: 209046.0
High threshold: 240740.0


In [5]:
# Step 2: Create binary labels for EV charging site suitability

def create_label(row):
    # Bad site conditions
    if row["flood_risk_proxy"] == "High":
        return 0
    
    if row["dist_road_m"] >= 100:
        return 0
    
    if row["dist_hospital_m"] >= 3000:
        return 0
    
    if row["population_level"] == "Low":
        return 0
    
    # Good site
    return 1

df["label"] = df.apply(create_label, axis=1)

print("\nLabel distribution:")
display(df["label"].value_counts().sort_index())

print("\nGood sites (label = 1):", (df["label"] == 1).sum())
print("Bad sites (label = 0):", (df["label"] == 0).sum())


Label distribution:


label
0    478
1    524
Name: count, dtype: int64


Good sites (label = 1): 524
Bad sites (label = 0): 478


In [6]:
# Step 3: Save labelled dataset

LABELLED_FILE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "labelled_sites.csv"
)

# Select columns for final labelled dataset
labelled_columns = [
    "location_id",
    "longitude",
    "latitude",
    "zone_name",
    "population_2011",
    "population_level",
    "dist_road_m",
    "dist_traffic_m",
    "dist_hospital_m",
    "dist_college_m",
    "dist_school_m",
    "dist_market_m",
    "dist_food_hotel_m",
    "dist_finance_m",
    "dist_vehicle_service_m",
    "dist_waterway_m",
    "dist_waterbody_m",
    "building_count_500m",
    "poi_count_500m",
    "health_poi_count_500m",
    "commercial_poi_count_500m",
    "land_use_class",
    "elevation_m",
    "slope_deg",
    "flood_risk_proxy",
    "label",
]

labelled_df = df[labelled_columns].copy()

labelled_df.to_csv(
    LABELLED_FILE,
    index=False,
    encoding="utf-8-sig"
)

print("\nLabelled dataset saved to:")
print(LABELLED_FILE)

print("\nLabelled dataset shape:", labelled_df.shape)

print("\nLabel distribution in saved file:")
display(labelled_df["label"].value_counts().sort_index())

print("\nFirst five labelled records:")
display(labelled_df.head())


Labelled dataset saved to:
C:\Users\jamad\Documents\geosense\data\processed\labelled_sites.csv

Labelled dataset shape: (1002, 26)

Label distribution in saved file:


label
0    478
1    524
Name: count, dtype: int64


First five labelled records:


,location_id,longitude,latitude,zone_name,population_2011,population_level,dist_road_m,dist_traffic_m,dist_hospital_m,dist_college_m,...,dist_waterbody_m,building_count_500m,poi_count_500m,health_poi_count_500m,commercial_poi_count_500m,land_use_class,elevation_m,slope_deg,flood_risk_proxy,label
0,1,73.756295,18.531983,Admin Ward 01 Aundh,180264.0,Low,63.912745,2554.453161,2656.654669,2756.862746,...,2355.857536,25,0,0,0,forest,657.0,13.665863,Low,0
1,2,73.761031,18.532014,Admin Ward 01 Aundh,180264.0,Low,8.326502,2182.850969,2236.843921,2342.377309,...,1867.756572,92,0,0,0,forest,631.0,5.602115,Low,0
2,3,73.760999,18.536532,Admin Ward 01 Aundh,180264.0,Low,96.469324,1854.429628,2179.448439,2064.879739,...,2047.747091,61,0,0,0,forest,662.0,10.661363,Low,0
3,4,73.766059,18.491383,Admin Ward 03 Kothrud Karveroad,209046.0,Low,7.424411,2405.992842,2412.172555,1286.193628,...,3355.323774,7,0,0,0,unknown,742.0,5.913284,Low,0
4,5,73.765995,18.500419,Admin Ward 03 Kothrud Karveroad,209046.0,Low,47.504002,1614.152389,1549.490810,476.130858,...,2507.593358,57,1,0,0,unknown,702.0,7.080391,Medium,0


In [7]:
from pathlib import Path

import joblib
import pandas as pd


PROJECT_ROOT = Path.cwd()

MODEL_FILE = PROJECT_ROOT / "models" / "saved" / "site_scorer_model.pkl"
LABELLED_FILE = PROJECT_ROOT / "data" / "processed" / "labelled_sites.csv"

model = joblib.load(MODEL_FILE)

df = pd.read_csv(LABELLED_FILE, encoding="utf-8-sig")

sample = df.head(10)

FEATURE_COLUMNS = [
    "dist_road_m",
    "dist_hospital_m",
    "dist_market_m",
    "dist_traffic_m",
    "building_count_500m",
    "poi_count_500m",
    "commercial_poi_count_500m",
    "population_2011",
    "elevation_m",
    "slope_deg",
]

df["flood_risk_encoded"] = df["flood_risk_proxy"].map(
    {"Low": 0, "Medium": 1, "High": 2}
).fillna(1)

X_sample = df[FEATURE_COLUMNS + ["flood_risk_encoded"]].head(10)
y_sample = df["label"].head(10)

predictions = model.predict(X_sample)
probabilities = model.predict_proba(X_sample)

print("Sample labels:")
print(y_sample.values)

print("\nModel predictions:")
print(predictions)

print("\nModel probabilities (class 0, class 1):")
print(probabilities)

print("\nProbability of class 1 (good site):")
print(probabilities[:, 1])

Sample labels:
[0 0 0 0 0 0 0 0 0 0]

Model predictions:
[0 0 0 0 0 0 0 0 0 0]

Model probabilities (class 0, class 1):
[[0.9988181  0.00118192]
 [0.99815816 0.00184187]
 [0.9988181  0.00118192]
 [0.99815816 0.00184187]
 [0.9981019  0.00189808]
 [0.9981012  0.0018988 ]
 [0.9989192  0.00108082]
 [0.99890465 0.00109536]
 [0.9981019  0.00189808]
 [0.9981669  0.00183306]]

Probability of class 1 (good site):
[0.00118192 0.00184187 0.00118192 0.00184187 0.00189808 0.0018988
 0.00108082 0.00109536 0.00189808 0.00183306]


In [8]:
good_sites = df[df["label"] == 1].copy()

good_sites = good_sites.sort_values(
    by=[
        "dist_road_m",
        "dist_hospital_m",
        "commercial_poi_count_500m",
        "poi_count_500m",
    ],
    ascending=[
        True,
        True,
        False,
        False,
    ]
)

print("Total good sites:", len(good_sites))

display(
    good_sites[
        [
            "location_id",
            "latitude",
            "longitude",
            "zone_name",
            "population_2011",
            "population_level",
            "dist_road_m",
            "dist_hospital_m",
            "poi_count_500m",
            "commercial_poi_count_500m",
            "flood_risk_proxy",
            "label",
        ]
    ].head(10)
)

Total good sites: 524


,location_id,latitude,longitude,zone_name,population_2011,population_level,dist_road_m,dist_hospital_m,poi_count_500m,commercial_poi_count_500m,flood_risk_proxy,label
991,992,18.555753,73.950357,Admin Ward 07 Nagar Road,238434.0,Medium,0.000116,412.453245,36,15,Medium,1
973,974,18.591846,73.940659,Admin Ward 06 Yerawda - Sangamwadi,499741.0,High,0.012773,2987.260171,1,0,Medium,1
661,662,18.487655,73.893945,Admin Ward 15 Kondhwa Wanavdi,270000.0,High,0.066304,526.149072,4,1,Medium,1
82,83,18.500541,73.784937,Admin Ward 04 Warje Karvenagar,232725.0,Medium,0.077329,771.576225,8,2,Low,1
967,968,18.564736,73.940826,Admin Ward 07 Nagar Road,238434.0,Medium,0.133243,217.424493,13,6,Medium,1
304,305,18.505328,73.827527,Admin Ward 04 Warje Karvenagar,232725.0,Medium,0.195194,224.966165,57,20,Medium,1
901,902,18.469799,73.931940,Admin Ward 13 Hadapsar,280215.0,High,0.408108,654.403170,1,0,Low,1
589,590,18.559862,73.879267,Admin Ward 06 Yerawda - Sangamwadi,499741.0,High,0.509672,606.950303,7,0,Medium,1
601,602,18.455971,73.884678,Admin Ward 15 Kondhwa Wanavdi,270000.0,High,0.511993,416.522522,9,4,Medium,1
976,977,18.496988,73.945979,Admin Ward 13 Hadapsar,280215.0,High,0.554244,286.757253,11,3,Medium,1


In [1]:
---------------------------------------------------------------------------
FileNotFoundError                         Traceback (most recent call last)
Cell In[2], line 11
      7 from sklearn.model_selection import train_test_split
      8 import shap
      9 
     10 # Load labeled data
---> 11 df = pd.read_parquet("data/interim/labeled_sites.parquet")
     12 
     13 # Define features
     14 feature_cols = [

File ~\.conda\envs\geosense\Lib\site-packages\pandas\io\parquet.py:671, in read_parquet(path, engine, columns, storage_options, dtype_backend, filesystem, filters, to_pandas_kwargs, **kwargs)
    668 impl = get_engine(engine)
    669 check_dtype_backend(dtype_backend)
--> 671 return impl.read(
    672     path,
    673     columns=columns,
    674     filters=filters,
    675     storage_options=storage_options,
    676     dtype_backend=dtype_backend,
    677     filesystem=filesystem,
    678     to_pandas_kwargs=to_pandas_kwargs,
    679     **kwargs,
    680 )

File ~\.conda\envs\geosense\Lib\site-packages\pandas\io\parquet.py:253, in PyArrowImpl.read(self, path, columns, filters, dtype_backend, storage_options, filesystem, to_pandas_kwargs, **kwargs)
    240 def read(
    241     self,
    242     path,
   (...)    249     **kwargs,
    250 ) -> DataFrame:
    251     kwargs["use_pandas_metadata"] = True
--> 253     path_or_handle, handles, filesystem = _get_path_or_handle(
    254         path,
    255         filesystem,
    256         storage_options=storage_options,
    257         mode="rb",
    258     )
    259     try:
    260         pa_table = self.api.parquet.read_table(
    261             path_or_handle,
    262             columns=columns,
   (...)    265             **kwargs,
    266         )

File ~\.conda\envs\geosense\Lib\site-packages\pandas\io\parquet.py:141, in _get_path_or_handle(path, fs, storage_options, mode, is_dir)
    131 handles = None
    132 if (
    133     not fs
    134     and not is_dir
   (...)    139     # fsspec resources can also point to directories
    140     # this branch is used for example when reading from non-fsspec URLs
--> 141     handles = get_handle(
    142         path_or_handle, mode, is_text=False, storage_options=storage_options
    143     )
    144     fs = None
    145     path_or_handle = handles.handle

File ~\.conda\envs\geosense\Lib\site-packages\pandas\io\common.py:939, in get_handle(path_or_buf, mode, encoding, compression, memory_map, is_text, errors, storage_options)
    930         handle = open(
    931             handle,
    932             ioargs.mode,
   (...)    935             newline="",
    936         )
    937     else:
    938         # Binary mode
--> 939         handle = open(handle, ioargs.mode)
    940     handles.append(handle)
    942 # Convert BytesIO or file objects passed with an encoding

FileNotFoundError: [Errno 2] No such file or directory: 'data/interim/labeled_sites.parquet'

IndentationError: unindent does not match any outer indentation level (<tokenize>, line 7)